In [1]:
import requests
import geopandas as gpd
import pandas as pd

wfs_url = "https://openmaps.gov.bc.ca/geo/pub/wfs"


In [2]:
params = {
    'service': 'WFS',
    'version': '2.0.0',
    'request': 'GetFeature',
    'typeName': 'pub:WHSE_LAND_AND_NATURAL_RESOURCE.PROT_HISTORICAL_FIRE_POLYS_SP',
    'outputFormat': 'application/json',
    'srsName': 'EPSG:4326',
    'CQL_FILTER': "FIRE_YEAR >= 2000 AND FIRE_YEAR <= 2024"
}
# filtering for fires from year 2000 to 2024

In [3]:
response = requests.get(wfs_url, params=params, timeout=600)

In [4]:

if response.status_code == 200:
    geojson_data = response.json()
    gdf = gpd.GeoDataFrame.from_features(geojson_data['features'])
    
    print(f"Total records (2000-2024): {len(gdf)}")
    print(f"\nColumns: {list(gdf.columns)}\n")
    
    # Filter for Cariboo Fire Centre
    # Cariboo fires have fire numbers starting with 'C'
    if 'FIRE_NUMBER' in gdf.columns:
        cariboo_gdf = gdf[gdf['FIRE_NUMBER'].str.startswith('C', na=False)].copy()
        print(f"Cariboo Fire Centre records: {len(cariboo_gdf)}")
    else:
        # Alternative: filter by Fire Centre field if available
        if 'FIRE_CENTRE' in gdf.columns:
            cariboo_gdf = gdf[gdf['FIRE_CENTRE'] == 'Cariboo'].copy()
        else:
            cariboo_gdf = gdf.copy()
            print("Warning: Could not identify FIRE_CENTRE field")
    
    # Extract date information for monthly analysis
    date_fields = ['FIRE_DATE', 'IGNITION_DATE', 'FIRE_START_DATE']
    date_field = None
    
    for field in date_fields:
        if field in cariboo_gdf.columns:
            date_field = field
            break
    
    if date_field:
        # Convert to datetime
        cariboo_gdf[date_field] = pd.to_datetime(cariboo_gdf[date_field], errors='coerce')
        cariboo_gdf['MONTH'] = cariboo_gdf[date_field].dt.month
        cariboo_gdf['MONTH_NAME'] = cariboo_gdf[date_field].dt.strftime('%B')
        print(f"\nExtracted month from {date_field}")
    
    # Calculate area in hectares
    area_fields = ['FIRE_SIZE_HECTARES', 'SIZE_HA', 'FEATURE_AREA_SQM']
    
    if 'FIRE_SIZE_HECTARES' not in cariboo_gdf.columns:
        if 'FEATURE_AREA_SQM' in cariboo_gdf.columns:
            cariboo_gdf['FIRE_SIZE_HECTARES'] = cariboo_gdf['FEATURE_AREA_SQM'] / 10000
        else:
            # Calculate from geometry
            cariboo_gdf['FIRE_SIZE_HECTARES'] = cariboo_gdf.geometry.area / 10000
    
    # Summary statistics
    if 'FIRE_YEAR' in cariboo_gdf.columns:
        print(f"\nYear range: {cariboo_gdf['FIRE_YEAR'].min()} to {cariboo_gdf['FIRE_YEAR'].max()}")
        
        yearly_summary = cariboo_gdf.groupby('FIRE_YEAR').agg({
            'FIRE_NUMBER': 'count',
            'FIRE_SIZE_HECTARES': 'sum'
        }).round(2)
        yearly_summary.columns = ['Fire_Count', 'Total_Hectares_Burned']
        print(f"\nYearly Summary:\n{yearly_summary}")
    
   
    
else:
    print(f"Error {response.status_code}: {response.text[:500]}")

Total records (2000-2024): 7186

Columns: ['geometry', 'FIRE_NUMBER', 'VERSION_NUMBER', 'FIRE_YEAR', 'FIRE_CAUSE', 'FIRE_LABEL', 'FIRE_SIZE_HECTARES', 'SOURCE', 'GPS_TRACK_DATE', 'LOAD_DATE', 'FIRE_DATE', 'CREATION_METHOD', 'FEATURE_CODE', 'OBJECTID', 'SE_ANNO_CAD_DATA', 'FEATURE_AREA_SQM', 'FEATURE_LENGTH_M']

Cariboo Fire Centre records: 1158

Extracted month from FIRE_DATE

Year range: 2000 to 2024

Yearly Summary:
           Fire_Count  Total_Hectares_Burned
FIRE_YEAR                                   
2000               15                  807.8
2001                1                  184.3
2002                1                  110.5
2003                7                31227.2
2004                7                30023.1
2005                2                  441.5
2006               27                33622.8
2007               99                  601.4
2008               89                 1805.6
2009              103               114123.0
2010              111               17

C:\Users\rajpu\AppData\Local\Temp\ipykernel_28336\3262453777.py:32: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  cariboo_gdf[date_field] = pd.to_datetime(cariboo_gdf[date_field], errors='coerce')


C:\Users\rajpu\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\pyogrio\geopandas.py:917: UserWarning: 'crs' was not provided.  The output dataset will not have projection information defined and may not be usable in other systems.
  write(


,FIRE_YEAR,FIRE_SIZE_HECTARES,OBJECTID,FEATURE_AREA_SQM,FEATURE_LENGTH_M
count,10000.000000,10000.00000,1.000000e+04,1.000000e+04,10000.000000
mean,1931.435500,575.33692,3.187958e+06,5.753863e+06,7353.987588
std,7.836784,2902.54934,2.912377e+03,2.902549e+07,12608.764139
min,1917.000000,0.10000,3.182943e+06,1.074211e+03,153.833900
25%,1925.000000,18.50000,3.185443e+06,1.855006e+05,1806.494100
50%,1931.000000,64.90000,3.187942e+06,6.497769e+05,3601.275300
75%,1938.000000,249.20000,3.190442e+06,2.492710e+06,7578.557650
max,2023.000000,132494.80000,3.193266e+06,1.324948e+09,187768.587600


C:\Users\rajpu\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\pyogrio\geopandas.py:917: UserWarning: 'crs' was not provided.  The output dataset will not have projection information defined and may not be usable in other systems.
  write(


Downloaded 10000 records

Available columns: ['geometry', 'VEG_FS_SYSID', 'FIRE_NUMBER', 'FIRE_YEAR', 'BURN_SEVERITY_RATING', 'PRE_FIRE_IMAGE', 'PRE_FIRE_IMAGE_DATE', 'POST_FIRE_IMAGE', 'POST_FIRE_IMAGE_DATE', 'AREA_HA', 'COMMENTS', 'FEATURE_AREA_SQM', 'FEATURE_LENGTH_M', 'OBJECTID', 'SE_ANNO_CAD_DATA']


C:\Users\rajpu\AppData\Local\Temp\ipykernel_40780\1438691907.py:12: UserWarning: Column names longer than 10 characters will be truncated when saved to ESRI Shapefile.
  gdf.to_file("bc_fire_burn_severity_all.shp", driver='ESRI Shapefile')
C:\Users\rajpu\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\pyogrio\geopandas.py:917: UserWarning: 'crs' was not provided.  The output dataset will not have projection information defined and may not be usable in other systems.
  write(
C:\Users\rajpu\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\pyogrio\raw.py:733: RuntimeWarning: Normalized/laundered field name: 'VEG_FS_SYSID' to 'VEG_FS_SYS'
  ogr_write(
C:\Users\rajpu\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\pyogrio\raw.py:733: RuntimeWarning: Normalized/launde


Data saved as shapefile: bc_fire_burn_severity_all.shp


C:\Users\rajpu\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\pyogrio\geopandas.py:917: UserWarning: 'crs' was not provided.  The output dataset will not have projection information defined and may not be usable in other systems.
  write(


Data saved as GeoJSON: bc_fire_burn_severity_all.geojson


Fetching BC Fire Perimeters (2000-2024)...

Total records (2000-2024): 7186

Columns: ['geometry', 'FIRE_NUMBER', 'VERSION_NUMBER', 'FIRE_YEAR', 'FIRE_CAUSE', 'FIRE_LABEL', 'FIRE_SIZE_HECTARES', 'SOURCE', 'GPS_TRACK_DATE', 'LOAD_DATE', 'FIRE_DATE', 'CREATION_METHOD', 'FEATURE_CODE', 'OBJECTID', 'SE_ANNO_CAD_DATA', 'FEATURE_AREA_SQM', 'FEATURE_LENGTH_M']

Cariboo Fire Centre records: 1158

Extracted month from FIRE_DATE

Year range: 2000 to 2024

Yearly Summary:
           Fire_Count  Total_Hectares_Burned
FIRE_YEAR                                   
2000               15                  807.8
2001                1                  184.3
2002                1                  110.5
2003                7                31227.2
2004                7                30023.1
2005                2                  441.5
2006               27                33622.8
2007               99                  601.4
2008               89                 1805.6
2009              103               114

C:\Users\rajpu\AppData\Local\Temp\ipykernel_23868\1501802852.py:53: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  cariboo_gdf[date_field] = pd.to_datetime(cariboo_gdf[date_field], errors='coerce')
C:\Users\rajpu\AppData\Local\Temp\ipykernel_23868\1501802852.py:80: UserWarning: Column names longer than 10 characters will be truncated when saved to ESRI Shapefile.
  cariboo_gdf.to_file("cariboo_fire_perimeters_2000_2024.shp", driver='ESRI Shapefile')
C:\Users\rajpu\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\pyogrio\geopandas.py:917: UserWarning: 'crs' was not provided.  The output dataset will not have projection information defined and may not be usable in other systems.
  write(
C:\Users\rajpu\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\Lo


✓ Files created:
  - cariboo_fire_perimeters_2000_2024.shp (for Tableau)
  - cariboo_fire_perimeters_2000_2024.geojson
  - cariboo_fire_perimeters_2000_2024.csv (tabular data)
